# Coordinate ordering


In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

REPO_ROOT = Path.cwd().parent

sys.path.insert(0, str(REPO_ROOT))

from utilities.function_library import modified_rosenbrock
from utilities.residual_orponet import ResidualOrPoNet
from utilities.training import Trainer


## 1. Settings


In [ ]:
DEVICE = torch.device("cpu")
DTYPE = torch.float32


## 2. Data and model


In [ ]:
DIMENSION = 10
DEGREE = 4
SAMPLES = 2000  # Per independent train/validation/test split.
SPLIT_SEEDS = {"train": 6100101, "validation": 6100102, "test": 6100103}


def data_split(split):
    engine = torch.quasirandom.SobolEngine(
        DIMENSION, scramble=True, seed=SPLIT_SEEDS[split]
    )
    physical_x = (2 * engine.draw(SAMPLES) - 1).to(dtype=DTYPE, device=DEVICE)
    return physical_x, modified_rosenbrock(physical_x)


def make_model(extra_constants):
    return ResidualOrPoNet(
        basis_configs=[
            {
                "type": "chebyshev",
                "degree": DEGREE,
                "constant_channels": 2 + extra_constants,
            }
            for _ in range(DIMENSION)
        ],
        data_domain_lb=[-1] * DIMENSION,
        data_domain_ub=[1] * DIMENSION,
        training_domain_lb=[-1] * DIMENSION,
        training_domain_ub=[1] * DIMENSION,
        coeff_init_std=0.1,
        neuron_copies=1,
        normalize_basis=False,
        use_additive_basis_branch=True,
        gate_config=None,
        dtype=DTYPE,
    ).to(DEVICE)


physical_train, train_y = data_split("train")
physical_validation, validation_y = data_split("validation")


## 3. Train both input orders


In [ ]:
SEED = 11


def train_case(extra_constants, order):

    indices = ORDERS[order]
    train_x = physical_train[:, indices]
    validation_x = physical_validation[:, indices]

    torch.manual_seed(SEED)
    model = make_model(extra_constants)
    model.apply_basis_aware_init(train_x, gain=0.1)

    mean, std = train_y.mean(), train_y.std()

    trainer = Trainer(
        model,
        (train_x, train_y),
        (validation_x, validation_y),
        target_stats=(mean, std),
        selection="validation_relative_l2",
    )

    loader = DataLoader(
        TensorDataset(train_x, trainer.train_targets),
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)

    trainer.fit(optimizer, epochs=500, batches=loader)

    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=20,
        history_size=100,
        tolerance_grad=1e-9,
        tolerance_change=1e-11,
        line_search_fn="strong_wolfe",
    )

    trainer.fit_lbfgs(optimizer, steps=20)

    trainer.restore_best()
    physical_test, test_y = data_split("test")

    prediction = model(physical_test[:, indices]) * std + mean
    # Promote before subtraction, matching the paper's direct test-MSE reduction.
    test_mse = (prediction.double() - test_y.double()).square().mean().item()

    return test_mse


In [ ]:
ORDERS = {
    "Adjacent": list(range(DIMENSION)),
    "Separated": [0, 2, 4, 6, 8, 1, 3, 5, 7, 9],
}

results = {order: [] for order in ORDERS}

for extra_constants in [0, 1, 2, 4, 8]:
    for order in ORDERS:
        mse = train_case(extra_constants, order)
        results[order].append(mse)
        print(
            f"extra constants={extra_constants}, {order}: test MSE={mse:.3e}",
            flush=True,
        )


extra constants=0, Adjacent: test MSE=1.386e-02
extra constants=0, Separated: test MSE=2.051e+00
extra constants=1, Adjacent: test MSE=3.988e-03
extra constants=1, Separated: test MSE=1.444e+00
extra constants=2, Adjacent: test MSE=5.509e-04
extra constants=2, Separated: test MSE=8.456e-01
extra constants=4, Adjacent: test MSE=1.395e-04
extra constants=4, Separated: test MSE=2.436e-02
extra constants=8, Adjacent: test MSE=2.288e-05
extra constants=8, Separated: test MSE=4.491e-06
